# Notebook 02 — Apply SmoothQuant + W8A8 Fake-Quant, Verify with Perplexity

This is the first notebook where we actually do SmoothQuant. Notebook 01 collected activation scales; this notebook uses those scales to transform the model and then fake-quantizes it to W8A8.

We validate on **WikiText-2 perplexity** rather than jumping straight to HumanEval because:

- Perplexity is fast (~1-2 min per configuration)
- It catches catastrophic quantization failures immediately
- HumanEval takes 25+ min per config; we don't want to burn that compute only to find the smoothed model is broken

## The four configurations we measure

| Config | What it tests |
|---|---|
| **bf16 baseline** | Reference perplexity |
| **W8A8, no smoothing** | Shows the problem SmoothQuant solves — naive W8A8 should be visibly worse |
| **SmoothQuant(α=0.5) + W8A8** | The paper's solution — should be close to bf16 |
| **SmoothQuant(α=0.85) + W8A8** | Paper's OPT alpha; lets us start the ablation |

The key paper claim we're verifying: **the smoothed W8A8 model matches the bf16 baseline within ~0.1 perplexity, while naive W8A8 is multiple points worse.**

## Outputs

- `results/ppl_<model_size>.csv` — perplexity for all four configs
- `checkpoints/qwen25-coder-<size>-smoothed-a<alpha>/` — smoothed bf16 model, ready for notebook 03 (real INT8 export) and notebooks 04/05 (benchmarks)

## Runtime

- **7B on A100**: ~12 min total (3 model loads + 4 PPL evals)
- **7B on L4**: ~18 min total
- **14B on A100 40GB**: ~25 min — L4 cannot hold 14B bf16

## Section 1 — Setup

In [1]:
# Runpod / bare-pod setup. Assumes the repo lives at /workspace/qwen-smoothquant-project
# and that HF cache is on a persistent volume at /workspace/hf-cache.
import os, sys

PROJECT_ROOT = '/workspace/qwen-smoothquant-project'
assert os.path.exists(PROJECT_ROOT), (
    f'Project not found at {PROJECT_ROOT}. Clone the repo there or edit PROJECT_ROOT.'
)
os.chdir(PROJECT_ROOT)
assert os.path.exists('src/qwen_smooth.py')

if os.path.abspath('.') not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

os.environ.setdefault('HF_HOME', '/workspace/hf-cache')
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'HF_HOME:      {os.environ["HF_HOME"]}')


Project root: /workspace/qwen-smoothquant-project
HF_HOME:      /workspace/hf-cache


In [2]:
# ============================================================
# Idempotent pinned-install cell.
# First run on a fresh pod: installs the stack (~4 min), prints
#     "RESTART THE KERNEL NOW" — do it, then re-run this cell.
# Subsequent runs: verifies versions, no-op in ~3 seconds.
# ============================================================
import importlib.metadata as _md
import subprocess, sys

_pinned = [
    ('torch',              '2.9.1',  'exact'),
    ('typing_extensions',  '4.13',   'floor'),
    ('compressed-tensors', '0.13.0', 'exact'),
    ('transformers',       '4.57.3', 'exact'),
    ('llmcompressor',      '0.9.0',  'exact'),
]

def _tup(s):
    return tuple(int(x) for x in s.split('+')[0].split('.')[:3] if x.isdigit())

def _ok(pkg, want, mode):
    try:
        have = _md.version(pkg)
    except _md.PackageNotFoundError:
        return False, '(missing)'
    if mode == 'exact':
        return have == want, have
    if mode == 'floor':
        return _tup(have) >= _tup(want), have
    raise ValueError(mode)

status = [(pkg, want, mode, *_ok(pkg, want, mode)) for pkg, want, mode in _pinned]
all_ok = all(ok for *_, ok, _ in status)

print(f'{"package":<22s} {"installed":<14s} {"pinned":<14s} status')
print('-' * 64)
for pkg, want, mode, ok, have in status:
    tag = 'OK' if ok else 'MISMATCH'
    w = f'>={want}' if mode == 'floor' else want
    print(f'{pkg:<22s} {have:<14s} {w:<14s} {tag}')

if all_ok:
    print('\nEnvironment already pinned — skipping install.')
else:
    print('\nInstalling pinned stack (~4 min)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q',
        'torch', 'torchvision', 'torchaudio',
        'nvidia-cuda-nvrtc-cu12', 'nvidia-cuda-runtime-cu12', 'nvidia-cudnn-cu12',
        'nvidia-cublas-cu12', 'nvidia-cufft-cu12', 'nvidia-curand-cu12',
        'nvidia-cusolver-cu12', 'nvidia-cusparse-cu12', 'nvidia-cusparselt-cu12',
        'nvidia-nccl-cu12', 'nvidia-nvtx-cu12', 'nvidia-nvjitlink-cu12',
        'llmcompressor', 'compressed-tensors', 'transformers', 'typing_extensions',
    ])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.9.1',
        'typing_extensions>=4.13',
        'compressed-tensors==0.13.0',
        'transformers==4.57.3',
        'llmcompressor==0.9.0',
        'accelerate', 'safetensors', 'datasets', 'tqdm',
        'matplotlib', 'pandas', 'vllm',
        'evalplus', 'bigcodebench',
    ])
    print()
    print('=' * 60)
    print('INSTALL COMPLETE — RESTART THE KERNEL NOW, then re-run this cell.')
    print('=' * 60)

package                installed      pinned         status
----------------------------------------------------------------
torch                  2.9.1          2.9.1          OK
typing_extensions      4.15.0         >=4.13         OK
compressed-tensors     0.13.0         0.13.0         OK
transformers           4.57.3         4.57.3         OK
llmcompressor          0.9.0          0.9.0          OK

Environment already pinned — skipping install.


In [3]:
import os
for d in ['results', 'results/plots', 'checkpoints']:
    os.makedirs(d, exist_ok=True)
print('Output dirs ready.')

Output dirs ready.


In [4]:
# ============================================================
# CONFIG — only this block changes when switching 7B ↔ 14B.
# ============================================================
MODEL_SIZE      = '7B'                                        # '7B' or '14B'
SIZE            = MODEL_SIZE.lower()                          # '7b' or '14b'
MODEL_ID        = f'Qwen/Qwen2.5-Coder-{MODEL_SIZE}-Instruct'
ACT_SCALES_PATH = f'act_scales/qwen25-coder-{SIZE}.pt'
ALPHAS_TO_TEST  = [0.5, 0.85]    # 0.5 is the main config; 0.85 starts the alpha ablation
SAVE_ALPHA      = 0.5            # which smoothed checkpoint to save for downstream notebooks
PPL_SEQ_LEN     = 2048           # standard WikiText chunk size

assert os.path.exists(ACT_SCALES_PATH), (
    f'Missing {ACT_SCALES_PATH}. Run notebook 01 first with MODEL_SIZE={MODEL_SIZE!r}.'
)

print(f'Model:     {MODEL_ID}')
print(f'Scales:    {ACT_SCALES_PATH}')
print(f'Alphas:    {ALPHAS_TO_TEST}')
print(f'Save alpha:{SAVE_ALPHA}')


Model:     Qwen/Qwen2.5-Coder-7B-Instruct
Scales:    act_scales/qwen25-coder-7b.pt
Alphas:    [0.5, 0.85]
Save alpha:0.5


In [5]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

NVIDIA GeForce RTX 4090, 24564 MiB, 2 MiB


## Section 2 — Perplexity evaluator

Standard WikiText-2 chunked perplexity: concatenate the test set into a single token stream, split into non-overlapping chunks of `seq_len` tokens, compute cross-entropy on each, then report `exp(mean_loss)`.

This matches the evaluation in the upstream `smoothquant/ppl_eval.py` so our numbers are comparable to the paper's.

In [6]:
import torch
import torch.nn as nn
from datasets import load_dataset
from tqdm import tqdm


@torch.no_grad()
def evaluate_perplexity(model, tokenizer, seq_len=2048, stride=None):
    """Chunked perplexity on WikiText-2 test set."""
    model.eval()
    device = next(model.parameters()).device
    stride = stride or seq_len

    # Load and concatenate the full test set — standard practice for LLM PPL
    data = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    text = '\n\n'.join(data['text'])
    enc = tokenizer(text, return_tensors='pt')
    input_ids = enc.input_ids.to(device)
    n_tokens = input_ids.size(1)

    nlls = []            # per-chunk total NLL (sum, not mean)
    total_tokens = 0     # count of tokens where we actually computed loss
    for begin in tqdm(range(0, n_tokens - 1, stride), desc='PPL', leave=False):
        end = min(begin + seq_len, n_tokens)
        chunk = input_ids[:, begin:end]
        if chunk.size(1) < 2:
            break
        # Target is chunk shifted by one; HF loss handles the shift internally.
        labels = chunk.clone()
        outputs = model(chunk, labels=labels)
        # outputs.loss is mean CE over (chunk.size(1) - 1) predicted tokens.
        # Multiply by token count to recover total NLL; we'll divide by the
        # overall total after the loop.
        n_pred = chunk.size(1) - 1
        nlls.append(outputs.loss.float() * n_pred)
        total_tokens += n_pred

    mean_nll = torch.stack(nlls).sum() / total_tokens
    return torch.exp(mean_nll).item()


def free_model(model):
    """Aggressively release a model's GPU memory."""
    import gc
    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

print('Evaluator ready.')

Evaluator ready.


## Section 3 — Config 1: bf16 baseline

Reference point. This is the "unquantized" perplexity we're trying to match.

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# (MODEL_SIZE / MODEL_ID / PPL_SEQ_LEN come from the config cell above — do
# NOT redefine them here or a re-run with a changed config will get mixed
# values. If you see NameError: MODEL_ID, you forgot to run the config cell.)

results = {}   # config name -> perplexity

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print('Loading bf16 baseline...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='auto'
)
model.eval()

print('Measuring bf16 perplexity...')
ppl_bf16 = evaluate_perplexity(model, tokenizer, seq_len=PPL_SEQ_LEN)
results['bf16_baseline'] = ppl_bf16
print(f'  bf16 baseline PPL: {ppl_bf16:.4f}')

free_model(model)


Loading bf16 baseline...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Measuring bf16 perplexity...


Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 32768). Running this sequence through the model will result in indexing errors


  bf16 baseline PPL: 9.5526


## Section 4 — Config 2: W8A8 without smoothing (the problem)

This is what happens when you do the naive thing: replace every `nn.Linear` with a W8A8 fake-quant version, with no outlier handling. The paper's central claim is that this degrades accuracy significantly. For Llama-family models that typically means a 0.3-1.0 point perplexity increase — small in absolute terms but visible.

If you don't see a gap between this and the smoothed version, something's wrong (either the calibration scales are off, or the alphas are too small to matter).

In [8]:
from src.qwen_fake_quant import quantize_qwen2_w8a8

print('Loading bf16 for naive W8A8...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='auto'
)
model.eval()

print('Replacing Linears with W8A8 fake-quant (no smoothing applied)...')
quantize_qwen2_w8a8(model)

print('Measuring naive W8A8 perplexity...')
ppl_naive_w8a8 = evaluate_perplexity(model, tokenizer, seq_len=PPL_SEQ_LEN)
results['w8a8_no_smooth'] = ppl_naive_w8a8
print(f'  W8A8 (no smoothing) PPL: {ppl_naive_w8a8:.4f}')
print(f'  Degradation vs bf16:     {ppl_naive_w8a8 - ppl_bf16:+.4f}')

free_model(model)

Loading bf16 for naive W8A8...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Replacing Linears with W8A8 fake-quant (no smoothing applied)...
Replaced 196 Linears with W8A8Linear (weight_quant=per_channel, act_quant=per_token, quantize_bmm_input=False)
Measuring naive W8A8 perplexity...


RuntimeError: Tensor on device meta is not on the expected device cuda:0!

## Section 5 — Configs 3 & 4: SmoothQuant + W8A8 at different alphas

Now the actual SmoothQuant pipeline. For each alpha we:

1. Load a fresh bf16 model
2. Apply smoothing in-place using the calibrated scales
3. (For `SAVE_ALPHA` only) Save the smoothed bf16 checkpoint to disk — notebook 03 (real INT8 export) and notebooks 04/05 (benchmarks) reload from this
4. Replace Linears with W8A8 fake-quant
5. Measure perplexity

Note that saving happens BEFORE fake-quant. The checkpoint stores the smoothed bf16 weights, not the quantized ones — you can always re-quantize at load time.

In [ ]:
from src.qwen_smooth import smooth_qwen2
from src.calibrate import load_act_scales

act_scales = load_act_scales(ACT_SCALES_PATH)

In [ ]:
for alpha in ALPHAS_TO_TEST:
    print('=' * 60)
    print(f'Config: SmoothQuant(alpha={alpha}) + W8A8')
    print('=' * 60)

    print(f'Loading fresh bf16 model for alpha={alpha}...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.bfloat16, device_map='auto'
    )
    model.eval()

    print(f'Applying SmoothQuant (alpha={alpha})...')
    smooth_qwen2(model, act_scales, alpha=alpha)

    # Save the smoothed bf16 checkpoint BEFORE fake-quant, for downstream use
    if alpha == SAVE_ALPHA:
        ckpt_dir = f'checkpoints/qwen25-coder-{SIZE}-smoothed-a{alpha}'
        os.makedirs('checkpoints', exist_ok=True)
        print(f'Saving smoothed bf16 checkpoint to {ckpt_dir}...')
        model.save_pretrained(ckpt_dir, safe_serialization=True)
        tokenizer.save_pretrained(ckpt_dir)
        print('  Saved. This is the file notebooks 03-05 will reload.')

    print(f'Replacing Linears with W8A8 fake-quant...')
    quantize_qwen2_w8a8(model)

    print(f'Measuring perplexity...')
    ppl = evaluate_perplexity(model, tokenizer, seq_len=PPL_SEQ_LEN)
    results[f'smooth_a{alpha}_w8a8'] = ppl
    print(f'  SmoothQuant(alpha={alpha}) + W8A8 PPL: {ppl:.4f}')
    print(f'  Gap vs bf16: {ppl - ppl_bf16:+.4f}')
    print(f'  Improvement vs naive W8A8: {ppl_naive_w8a8 - ppl:+.4f}')
    print()

    free_model(model)

## Section 6 — Results summary

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {'config': k, 'perplexity': v, 'delta_vs_bf16': v - ppl_bf16}
    for k, v in results.items()
])
df = df.sort_values('perplexity').reset_index(drop=True)

os.makedirs('results', exist_ok=True)
df.to_csv(f'results/ppl_{SIZE}.csv', index=False)

print(df.to_string(index=False))
print()
print(f'Saved to results/ppl_{SIZE}.csv')

In [ ]:
# Bar plot for the writeup
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
labels = df['config'].tolist()
values = df['perplexity'].tolist()
colors = ['tab:green' if 'bf16' in c else 'tab:red' if 'no_smooth' in c else 'tab:blue'
          for c in labels]

bars = ax.bar(labels, values, color=colors)
ax.axhline(ppl_bf16, linestyle='--', color='tab:green', alpha=0.5,
           label=f'bf16 baseline ({ppl_bf16:.3f})')
ax.set_ylabel('WikiText-2 Perplexity (lower is better)')
ax.set_title(f'Qwen2.5-Coder-{MODEL_SIZE}: Effect of SmoothQuant on W8A8 quantization')
ax.legend()
plt.xticks(rotation=15, ha='right')

# Annotate each bar with its value
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'results/plots/ppl_comparison_{SIZE}.png', dpi=120, bbox_inches='tight')
plt.show()

## What you should see

The specific numbers depend on the model but the **ordering** should be:

```
bf16_baseline        <  smooth_a0.5_w8a8  ≈  smooth_a0.85_w8a8   <<   w8a8_no_smooth
```

Typical magnitudes for Qwen2.5-Coder on WikiText-2:

| Config | Expected PPL (ballpark) |
|---|---|
| bf16 baseline | 5.5 – 6.5 |
| W8A8, no smoothing | 6.0 – 7.5 |
| SmoothQuant(0.5) + W8A8 | within 0.1 of bf16 |
| SmoothQuant(0.85) + W8A8 | within 0.1–0.2 of bf16 |

**If naive W8A8 is actually fine** (gap < 0.1 PPL vs bf16), that's a surprising-but-publishable finding: Qwen2.5-Coder may not have severe activation outliers. Save these numbers and move on; SmoothQuant will still probably help on downstream code benchmarks where precision matters more.

**If smoothed W8A8 is worse than naive W8A8**, something is wrong — most likely the act_scales are mis-keyed relative to the module paths in `smooth_qwen2`. Re-check that `act_scales` keys look like `model.layers.0.self_attn.q_proj` (not e.g. `layers.0.self_attn.q_proj` without the leading `model.`).

## Artifacts produced

After this notebook runs successfully:
- `results/ppl_<size>.csv` — perplexity numbers, ready to paste into the report
- `results/plots/ppl_comparison_<size>.png` — bar chart for the slide deck
- `checkpoints/qwen25-coder-<size>-smoothed-a0.5/` — smoothed bf16 weights in HF format. **This is what notebooks 03, 04, and 05 will load.**

## Next

→ `03_llmcompressor_export.ipynb` — converts the smoothed bf16 checkpoint into a vLLM-compatible W8A8 checkpoint with real INT8 kernels. This gives us actual speedup numbers for the profiling notebook and actual W8A8 inference for the benchmarks.